#BASIC

1
- warehouse 
    - warehouse is a storage where the data is stored in structured format.

- lake
   - lake is a storage layer where the data(structured and unstructured) can be stored in any format

- lakehouse
   - lakehouse is the combination of both warehouse and lake which used to store both structured and unstructured data.It provide both - the cheapest price and data governance.it also have features like ACID and schema evaluation.

In [0]:
%sql
CREATE OR REPLACE TABLE dev.bronze.product(
    product_id LONG,
    name STRING,
    category STRING,
    price DOUBLE
)

In [0]:
%sql
INSERT INTO dev.bronze.product(product_id, name, category, price) VALUES
(1, 'Laptop', 'Electronic', 10000.0),
(2, 'Phone', 'Electronic', 5000.0),
(3, 'Tablet', 'Electronic', 3000.0),
(4, 'Shirt', 'Clothing', 50.0),
(5, 'Pants', 'Clothing', 100.0),
(6, 'Shoes', 'Clothing', 200.0),
(7, 'Pen', 'Stationary', 10.0),
(8, 'Notebook', 'Stationary', 15.0),
(9, 'Chair', 'Furniture', 1500.0),
(10, 'Table', 'Furniture', 3000.0)

In [0]:
%sql
INSERT INTO dev.bronze.product(product_id, name, category, price) VALUES
(11,'Bat','Sports',500.0),
(12,'Ball','Sports',200.0)

In [0]:
%sql
UPDATE dev.bronze.product SET price = 10600.0 WHERE product_id = 1

In [0]:
%sql
DESC HISTORY dev.bronze.product

In [0]:
%sql
SELECT * FROM dev.bronze.product version AS OF 2

-- in version 0 - no data 
-- in version 1 - insert data (10 rows)
-- in version 2 - add 2 more rows (12 rows)
-- in version 3 - update price of product_id = 1

#INTERMEDIATE

In [0]:
%sql
INSERT INTO dev.bronze.product(product_id, name, category, price,color) VALUES
(13,'Bike','Sports',3000.0,'Blue')

In [0]:
from pyspark.sql import Row

data = [(3,'Bike','Sports',3000.0,'Blue')]
schema = ["product_id","name","category","price","color"]
df = spark.createDataFrame(data,schema)
df.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("dev.bronze.product")

In [0]:
%sql
DESC HISTORY dev.bronze.product

In [0]:
%sql
SELECT * FROM dev.bronze.product TIMESTAMP AS OF '2026-08-21T04:57:08.000+00:00'

In [0]:
%sql
SELECT * FROM dev.bronze.product VERSION AS OF 3

In [0]:
%sql
RESTORE TABLE dev.bronze.product
TO VERSION AS OF 3

7.
- ACID transaction matters the most because it prevent to store corrupt data - if the transaction(insert,update,delete) fails or partially complete then the changes can be roll back to previous state and keep the table consistent.

- When multiple pipelines update the same data concurrently - delta lake detects conflict changes and prevent to commit both at same time.


#ADVANCED

8. 
- in nightly batch warehouse load , if the loads/transaction fails or partially committed then , warehouse stores the stale data or incomplete data which affects the business analysis results.

- with lakehouse pipelines which follows ACID transactions it prevent corrupt data to store - even the transaction fails or partial complete ,it not commited that change which provide consistent data.
If by mistake the data is deleted - it provide an option of time travelling means data can be restore to the previous or earlier state which reduce the operational risk.

ACID transactions and time travel make lakehouse pipelines more reliable, recoverable, and safer to operate than a traditional nightly batch load.


In [0]:
%sql
UPDATE dev.bronze.product SET price = 2000.0 WHERE product_id = 1
 

In [0]:
%sql
DESC HISTORY dev.bronze.product

10. 

- Lakehouse provide several advantages over warehouse - 
1. A lakehouse can store data in any format like csv,json etc which provide a feature to anlayst to work with any type of data not stick to one format or no conversion needed.

2. While creating dashboards/charts - analyst can do analysis on changing reports results . lakehouse provide the feature of time travel using version and timestamp using this analyst can get previous data and can compare with current results and do analysis.

3. Relaiable data - In lakehouse the data is consistent so it easy to analyze and the result is highly accurate.

- tradeoff - 
analyst should be aware or should have undestanding of the concepts of delta like time travel,version and timestamp .